# Collect: yearly DoD dollars by category and branch

`award_amount` in the award records is a lifetime total, so it can't say how much a company was paid in a given year. The `spending_by_category` endpoints return the dollars actually obligated in the time period searched, with no 500-award limit. This notebook pulls those totals for each company and fiscal year, grouped by product and service code (PSC) and by awarding branch.

In [1]:
import json
import os
import time

import requests
import pandas as pd

HEADERS = {"User-Agent": "WebDataScience/1.0 (INFO 4617; brkr6998@colorado.edu)"}

CATEGORY_URL = "https://api.usaspending.gov/api/v2/search/spending_by_category"

COMPANIES = {
    "Lockheed Martin": ["Lockheed Martin"],
    "RTX": ["Raytheon", "RTX Corporation", "Pratt & Whitney", "Collins Aerospace", "Rockwell Collins"],
    "Boeing": ["Boeing"],
    "General Dynamics": ["General Dynamics", "Electric Boat", "Bath Iron Works"],
    "Northrop Grumman": ["Northrop Grumman"],
    "Palantir": ["Palantir"],
    "Anduril": ["Anduril"],
    "Shield AI": ["Shield AI"],
    "Scale AI": ["Scale AI"],
}

FISCAL_YEARS = range(2020, 2027)
GROUPINGS = ["psc", "awarding_subagency"]

os.makedirs("data/raw/totals", exist_ok=True)

In [2]:
def get_page(company, fiscal_year, grouping, page):
    """Return one page of category totals, using the saved copy if it was already pulled."""
    path = f"data/raw/totals/{company.replace(' ', '_')}_FY{fiscal_year}_{grouping}_p{page}.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)

    body = {
        "filters": {
            "agencies": [{"type": "awarding", "tier": "toptier", "name": "Department of Defense"}],
            "time_period": [{"start_date": f"{fiscal_year - 1}-10-01", "end_date": f"{fiscal_year}-09-30"}],
            "award_type_codes": ["A", "B", "C", "D"],
            "recipient_search_text": COMPANIES[company],
        },
        "limit": 100,
        "page": page,
    }
    response = requests.post(f"{CATEGORY_URL}/{grouping}/", json=body, headers=HEADERS, timeout=120)
    response.raise_for_status()
    data = response.json()

    with open(path, "w") as f:
        json.dump(data, f)
    time.sleep(1)
    return data


def get_totals(company, fiscal_year, grouping):
    """Page through every category for one company, fiscal year, and grouping."""
    rows = []
    page = 1
    while True:
        data = get_page(company, fiscal_year, grouping, page)
        rows.extend(data["results"])
        if not data["page_metadata"]["hasNext"]:
            break
        page += 1
    return rows

## Pull every company, year, and grouping

In [3]:
all_rows = []
for company in COMPANIES:
    for fiscal_year in FISCAL_YEARS:
        for grouping in GROUPINGS:
            for row in get_totals(company, fiscal_year, grouping):
                all_rows.append({
                    "company": company,
                    "fiscal_year": fiscal_year,
                    "grouping": "psc" if grouping == "psc" else "branch",
                    "code": row["code"],
                    "name": row["name"],
                    "amount": row["amount"],
                })
    print(f"{company:18} done, {len(all_rows):,} rows so far", flush=True)

totals = pd.DataFrame(all_rows)
totals.to_csv("data/category_totals.csv", index=False)
print(totals.shape)
print(totals.head().to_string())

Lockheed Martin    done, 2,429 rows so far


RTX                done, 4,899 rows so far


Boeing             done, 7,194 rows so far


General Dynamics   done, 9,650 rows so far


Northrop Grumman   done, 11,981 rows so far


Palantir           done, 12,066 rows so far


Anduril            done, 12,201 rows so far


Shield AI          done, 12,234 rows so far


Scale AI           done, 12,264 rows so far


(12264, 6)
           company  fiscal_year grouping  code                                               name        amount
0  Lockheed Martin         2020      psc  1510                               AIRCRAFT, FIXED WING  3.914854e+10
1  Lockheed Martin         2020      psc  1410                                    GUIDED MISSILES  8.679648e+09
2  Lockheed Martin         2020      psc  1520                              AIRCRAFT, ROTARY WING  3.889813e+09
3  Lockheed Martin         2020      psc  1905                   COMBAT SHIPS AND LANDING VESSELS  2.447284e+09
4  Lockheed Martin         2020      psc  1680  MISCELLANEOUS AIRCRAFT ACCESSORIES AND COMPONENTS  1.678347e+09


Both groupings cover the same dollars, so each company-year should add up to the same amount either way.

In [4]:
check = totals.pivot_table(index=["company", "fiscal_year"], columns="grouping", values="amount", aggfunc="sum")
check["difference"] = (check["psc"] - check["branch"]).abs()
print("Largest difference between groupings:", round(check["difference"].max(), 2))
print((check["psc"].unstack() / 1e9).round(2).to_string())

Largest difference between groupings: 0.03
fiscal_year        2020   2021   2022   2023   2024   2025   2026
company                                                          
Anduril            0.03   0.01   0.11   0.16   0.58   0.46   0.66
Boeing            24.20  25.27  17.16  24.22  24.75  23.38  32.60
General Dynamics  24.35  20.15  26.16  28.72  24.09  33.96  25.33
Lockheed Martin   76.66  41.19  48.01  72.76  52.90  76.61  56.91
Northrop Grumman  12.46  13.01  13.89  16.14  19.10  13.70  13.92
Palantir           0.14   0.08   0.23   0.18   0.36   0.79   1.68
RTX               27.22  20.46  25.73  31.55  27.97  36.66  35.88
Scale AI           0.02   0.00   0.04   0.02   0.03   0.01   0.01
Shield AI          0.00   0.00   0.02   0.01   0.00   0.01   0.01
